In [13]:
import pandas as pd

In [14]:
response_df = pd.read_json("Dataset/response.jsonl", lines=True)

In [15]:
response_df.head()

,id,source_id,model,temperature,labels,split,quality,response
0,0,15592,gpt-4-0613,0.7,[],train,good,The Anne Frank House has revealed that Anne Fr...
1,1,15592,gpt-3.5-turbo-0613,0.7,[],train,good,New research released by the Anne Frank House ...
2,2,15592,mistral-7B-instruct,0.7,"[{'start': 636, 'end': 653, 'text': 'February ...",train,good,New research conducted by the Anne Frank House...
3,3,15592,llama-2-7b-chat,0.7,"[{'start': 693, 'end': 754, 'text': 'Anne's di...",train,good,Here is a summary of the article in 115 words ...
4,4,15592,llama-2-13b-chat,0.7,[],train,good,"Seventy years ago, Anne Frank died of typhus i..."


In [16]:
source_df = pd.read_json("Dataset/source_info.jsonl", lines=True)

In [17]:
source_df.head()

,source_id,task_type,source,source_info,prompt
0,15592,Summary,CNN/DM,"Seventy years ago, Anne Frank died of typhus i...",Summarize the following news within 116 words:...
1,15593,Summary,CNN/DM,"Andrew Getty, one of the heirs to billions of ...",Summarize the following news within 89 words:\...
2,15594,Summary,CNN/DM,"Since Iran's Islamic Revolution in 1979, women...",Summarize the following news within 98 words:\...
3,15595,Summary,CNN/DM,"It's the kind of thing you see in movies, like...",Summarize the following news within 200 words:...
4,15596,Summary,CNN/DM,The FBI charged a Philadelphia woman on Thursd...,Summarize the following news within 86 words:\...


In [18]:
print(response_df.shape)
print(source_df.shape)

(17790, 8)
(2965, 5)


In [19]:
print(response_df.columns.tolist())

['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response']


In [20]:
print(source_df.columns.tolist())

['source_id', 'task_type', 'source', 'source_info', 'prompt']


In [21]:
response_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17790 entries, 0 to 17789
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           17790 non-null  int64  
 1   source_id    17790 non-null  int64  
 2   model        17790 non-null  str    
 3   temperature  17790 non-null  float64
 4   labels       17790 non-null  object 
 5   split        17790 non-null  str    
 6   quality      17790 non-null  str    
 7   response     17790 non-null  str    
dtypes: float64(1), int64(2), object(1), str(4)
memory usage: 1.1+ MB


In [22]:
source_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2965 entries, 0 to 2964
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   source_id    2965 non-null   int64 
 1   task_type    2965 non-null   str   
 2   source       2965 non-null   str   
 3   source_info  2965 non-null   object
 4   prompt       2965 non-null   str   
dtypes: int64(1), object(1), str(3)
memory usage: 115.9+ KB


In [23]:
response_df['model'].unique()

<StringArray>
[         'gpt-4-0613',  'gpt-3.5-turbo-0613', 'mistral-7B-instruct',
     'llama-2-7b-chat',    'llama-2-13b-chat',    'llama-2-70b-chat']
Length: 6, dtype: str

In [24]:
source_df['task_type'].unique()

<StringArray>
['Summary', 'Data2txt', 'QA']
Length: 3, dtype: str

In [25]:
source_df['source'].unique()

<StringArray>
['CNN/DM', 'Recent News', 'Yelp', 'MARCO']
Length: 4, dtype: str

In [26]:
combined_df = response_df.merge(
    source_df,
    on="source_id",
    how="left"
)

print("Combined dataset shape:", combined_df.shape)
print(combined_df.columns.tolist())

Combined dataset shape: (17790, 12)
['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'task_type', 'source', 'source_info', 'prompt']


In [27]:
print("Number of responses with labels:",
      (response_df["labels"].apply(len) > 0).sum())

print("Number of responses without labels:",
      (response_df["labels"].apply(len) == 0).sum())

Number of responses with labels: 7664
Number of responses without labels: 10126


In [28]:
label_types = []

for labels in response_df["labels"]:
    for label in labels:
        label_types.append(label["label_type"])

print(pd.Series(label_types).value_counts())

Evident Baseless Info    6237
Evident Conflict         5324
Subtle Baseless Info     2527
Subtle Conflict           201
Name: count, dtype: int64


In [29]:
combined_df["target"] = combined_df["labels"].apply(
    lambda x: 1 if len(x) > 0 else 0
)

In [30]:
print(combined_df["target"].value_counts())

target
0    10126
1     7664
Name: count, dtype: int64


In [31]:
combined_df[["response", "labels", "target"]].head(10)

,response,labels,target
0,The Anne Frank House has revealed that Anne Fr...,[],0
1,New research released by the Anne Frank House ...,[],0
2,New research conducted by the Anne Frank House...,"[{'start': 636, 'end': 653, 'text': 'February ...",1
3,Here is a summary of the article in 115 words ...,"[{'start': 693, 'end': 754, 'text': 'Anne's di...",1
4,"Seventy years ago, Anne Frank died of typhus i...",[],0
5,New research suggests that Anne Frank and her ...,"[{'start': 348, 'end': 388, 'text': 'sisters d...",1
6,"Andrew Getty, oil tycoon J. Paul Getty's grand...",[],0
7,"Andrew Getty, one of the heirs to billions of ...",[],0
8,"Andrew Getty, one of the heirs to billions of ...",[],0
9,"Andrew Getty, a billionaire heir to the Getty ...",[],0


In [32]:
print(combined_df["split"].value_counts())

split
train    15090
test      2700
Name: count, dtype: int64


In [33]:
print(
    combined_df.groupby("split")["source_id"]
    .nunique()
)

split
test      450
train    2515
Name: source_id, dtype: int64


In [34]:
source_split_check = combined_df.groupby("source_id")["split"].nunique()

print("Sources appearing in multiple splits:",
      (source_split_check > 1).sum())

Sources appearing in multiple splits: 0


In [35]:
print(
    "Duplicate responses:",
    combined_df["response"].duplicated().sum()
)

Duplicate responses: 238


In [36]:
print(
    "Duplicate source-model-response records:",
    combined_df[
        ["source_id", "model", "temperature", "response"]
    ].duplicated().sum()
)

Duplicate source-model-response records: 0


In [37]:
duplicate_responses = combined_df[
    combined_df["response"].duplicated(keep=False)
].sort_values("response")

duplicate_responses[
    ["source_id", "model", "temperature", "response", "target"]
].head(20)

,source_id,model,temperature,response,target
14438,15327,mistral-7B-instruct,0.775,"Based on the given passages, the dog breed ass...",0
14440,15327,llama-2-13b-chat,0.775,"Based on the given passages, the dog breed ass...",0
16153,12183,gpt-3.5-turbo-0613,0.700,Republicans use a combination of winner-take-a...,0
16152,12183,gpt-4-0613,0.700,Republicans use a combination of winner-take-a...,0
3707,12084,llama-2-70b-chat,1.000,The National Weather Service predicts tornadoe...,0
3713,12085,llama-2-70b-chat,1.000,The National Weather Service predicts tornadoe...,0
15961,15582,gpt-3.5-turbo-0613,0.700,The passages do not contain the necessary info...,0
17743,12453,gpt-3.5-turbo-0613,0.700,The passages do not contain the necessary info...,0
16158,12184,gpt-4-0613,0.700,Unable to answer based on given passages.,0
16362,12219,gpt-4-0613,0.700,Unable to answer based on given passages.,0


In [38]:
print(
    "Empty responses:",
    (combined_df["response"].str.strip() == "").sum()
)

Empty responses: 0


In [39]:
response_lengths = combined_df["response"].str.len()

In [40]:
print("Shortest response:", response_lengths.min())
print("Longest response:", response_lengths.max())
print("Average response length:", response_lengths.mean())

Shortest response: 7
Longest response: 4007
Average response length: 797.6752670039348


In [41]:
print("\nQuality distribution:")
print(combined_df["quality"].value_counts())


Quality distribution:
quality
good                 17617
incorrect_refusal      144
truncated               29
Name: count, dtype: int64


In [42]:
print(
    pd.crosstab(
        combined_df["quality"],
        combined_df["target"]
    )
)

target                0     1
quality                      
good               9976  7641
incorrect_refusal   143     1
truncated             7    22


In [43]:
combined_df[
    combined_df["quality"] != "good"
][
    ["source_id", "quality", "response", "labels", "target"]
].head(10)

,source_id,quality,response,labels,target
2132,11819,truncated,"Laurence Rasti, a Swiss-born Iranian photograp...","[{'start': 144, 'end': 163, 'text': '80 Irania...",1
2186,11829,truncated,"Dzhokhar ""Jahar"" Tsarnaev's defense team is ap...","[{'start': 228, 'end': 262, 'text': 'who was k...",1
4760,13964,incorrect_refusal,Tracking Flu in the USA: A Look at Texas,[],0
5925,13649,truncated,"Based on the provided structured data, here is...","[{'start': 465, 'end': 561, 'text': 'Some cust...",1
10054,14775,truncated,Here is an objective overview of the local bus...,"[{'start': 516, 'end': 527, 'text': 'Music: No...",1
10136,14790,truncated,Our Daily Bread is a restaurant located in San...,"[{'start': 186, 'end': 206, 'text': 'operates ...",1
11906,14300,incorrect_refusal,Unable to answer based on given passages.,[],0
11911,14301,incorrect_refusal,Unable to answer based on given passages.,[],0
12043,14323,incorrect_refusal,Unable to answer based on given passages.,[],0
12207,14351,truncated,To slow and tender cook a chuck roast in the o...,"[{'start': 754, 'end': 767, 'text': '350°F (17...",1


In [44]:
model_df = combined_df[combined_df["quality"] == "good"].copy()

In [45]:
model_df.head(2)

,id,source_id,model,temperature,labels,split,quality,response,task_type,source,source_info,prompt,target
0,0,15592,gpt-4-0613,0.7,[],train,good,The Anne Frank House has revealed that Anne Fr...,Summary,CNN/DM,"Seventy years ago, Anne Frank died of typhus i...",Summarize the following news within 116 words:...,0
1,1,15592,gpt-3.5-turbo-0613,0.7,[],train,good,New research released by the Anne Frank House ...,Summary,CNN/DM,"Seventy years ago, Anne Frank died of typhus i...",Summarize the following news within 116 words:...,0


In [46]:
print("Modeling dataset shape:", model_df.shape)

Modeling dataset shape: (17617, 13)


In [47]:
print("\nQuality:")
print(model_df["quality"].value_counts())


Quality:
quality
good    17617
Name: count, dtype: int64


In [48]:
print("\nTarget:")
print(model_df["target"].value_counts())


Target:
target
0    9976
1    7641
Name: count, dtype: int64


In [49]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [50]:
model_df["source_info"] = model_df["source_info"].apply(clean_text)

In [51]:
model_df["response"] = model_df["response"].apply(clean_text)

In [52]:
model_df[["source_info", "response", "target"]].head(3)

,source_info,response,target
0,"Seventy years ago, Anne Frank died of typhus i...",The Anne Frank House has revealed that Anne Fr...,0
1,"Seventy years ago, Anne Frank died of typhus i...",New research released by the Anne Frank House ...,0
2,"Seventy years ago, Anne Frank died of typhus i...",New research conducted by the Anne Frank House...,1


In [53]:
model_df["model_input"] = (
    model_df["source_info"] + " [SEP] " + model_df["response"]
)

In [54]:
model_df[["model_input", "target"]].head(3)

,model_input,target
0,"Seventy years ago, Anne Frank died of typhus i...",0
1,"Seventy years ago, Anne Frank died of typhus i...",0
2,"Seventy years ago, Anne Frank died of typhus i...",1


In [55]:
print(model_df["split"].value_counts())

split
train    14942
test      2675
Name: count, dtype: int64


In [56]:
print(pd.crosstab(model_df["split"], model_df["target"]))

target     0     1
split             
test    1732   943
train   8244  6698


In [57]:
X_train = model_df[model_df["split"] == "train"]["model_input"]
y_train = model_df[model_df["split"] == "train"]["target"]

In [58]:
X_test = model_df[model_df["split"] == "test"]["model_input"]
y_test = model_df[model_df["split"] == "test"]["target"]

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

In [60]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [61]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (14942, 5000)
Testing TF-IDF shape: (2675, 5000)


In [62]:
from sklearn.linear_model import LogisticRegression

In [63]:
log_reg = LogisticRegression(max_iter=1000)

In [64]:
log_reg.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [65]:
y_pred = log_reg.predict(X_test_tfidf)

In [66]:
print("Predictions generated:", len(y_pred))

Predictions generated: 2675


In [67]:
from sklearn.metrics import accuracy_score, classification_report

In [68]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.7685981308411215

              precision    recall  f1-score   support

           0       0.80      0.86      0.83      1732
           1       0.70      0.60      0.65       943

    accuracy                           0.77      2675
   macro avg       0.75      0.73      0.74      2675
weighted avg       0.76      0.77      0.76      2675



In [69]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1492  240]
 [ 379  564]]


In [70]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[1492  240]
 [ 379  564]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.86      0.83      1732
           1       0.70      0.60      0.65       943

    accuracy                           0.77      2675
   macro avg       0.75      0.73      0.74      2675
weighted avg       0.76      0.77      0.76      2675



In [71]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

In [72]:
nb_model.fit(X_train_tfidf, y_train)

print("Naive Bayes trained successfully!")

Naive Bayes trained successfully!


In [73]:
y_pred_nb = nb_model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred_nb))

Predictions generated: 2675


In [74]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)


In [75]:
print("Naive Bayes Accuracy:", accuracy_nb)
print()
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.7439252336448599

              precision    recall  f1-score   support

           0       0.79      0.81      0.80      1732
           1       0.64      0.61      0.63       943

    accuracy                           0.74      2675
   macro avg       0.72      0.71      0.72      2675
weighted avg       0.74      0.74      0.74      2675



In [76]:
from sklearn.svm import SVC

svm_model = SVC(kernel="linear")

In [77]:
svm_model.fit(X_train_tfidf, y_train)

print("SVM trained successfully!")

SVM trained successfully!


In [78]:
y_pred_svm = svm_model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred_svm))

Predictions generated: 2675


In [79]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)

In [80]:
print("SVM Accuracy:", accuracy_svm)
print()
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.7779439252336449

              precision    recall  f1-score   support

           0       0.79      0.89      0.84      1732
           1       0.74      0.57      0.64       943

    accuracy                           0.78      2675
   macro avg       0.77      0.73      0.74      2675
weighted avg       0.77      0.78      0.77      2675



In [81]:
from sklearn.ensemble import RandomForestClassifier

In [82]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [83]:
rf_model.fit(X_train_tfidf, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [84]:
y_pred_rf = rf_model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred_rf))

Predictions generated: 2675


In [85]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)


In [86]:
print("Random Forest Accuracy:", accuracy_rf)
print()
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7809345794392524

              precision    recall  f1-score   support

           0       0.80      0.89      0.84      1732
           1       0.74      0.59      0.65       943

    accuracy                           0.78      2675
   macro avg       0.77      0.74      0.75      2675
weighted avg       0.78      0.78      0.77      2675



In [88]:
from scipy.sparse import save_npz

save_npz("Processed data/X_train_tfidf.npz", X_train_tfidf)
save_npz("Processed data/X_test_tfidf.npz", X_test_tfidf)

In [90]:
import numpy as np

np.save("Processed data/y_train.npy", y_train)
np.save("Processed data/y_test.npy", y_test)

In [92]:
import joblib

joblib.dump(tfidf, "Processed data/tfidf_vectorizer.pkl")

['Processed data/tfidf_vectorizer.pkl']